[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fisica_de_Plasmas_y_Fusion_%28PLA%29/PLA-11_Plasmas_Tokamak_Orbits.ipynb)


# Investigación Avanzada: Plasmas Tokamak Orbits

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Guiding center drift orbits in a simplified Tokamak magnetic field
# Toroidal geometry (r, theta, phi)

R0 = 3.0   # Major radius (m)
B0 = 2.0   # On-axis toroidal magnetic field (T)
q = 2.0    # Safety factor (assumed constant for simplicity)
m = 1.67e-27 # Proton mass (kg)
e = 1.6e-19  # Elementary charge (C)
E = 10e3 * e # Particle energy 10 keV
mu = E / B0  # Magnetic moment (approx constant)

def magnetic_field(r, theta):
    # Simplified B field model
    B_phi = B0 * R0 / (R0 + r * np.cos(theta))
    B_theta = B_phi * r / (q * R0)
    B = np.sqrt(B_phi**2 + B_theta**2)
    return B_phi, B_theta, B

def guiding_center_drift(t, y):
    r, theta, phi = y
    B_phi, B_theta, B = magnetic_field(r, theta)
    
    # Grad-B and curvature drifts (simplified for passing/trapped particles)
    v_parallel = np.sqrt(2 * max(0, E - mu * B) / m)
    
    # Simple passing particle approximation
    dphi_dt = v_parallel * B_phi / (B * (R0 + r * np.cos(theta)))
    dtheta_dt = v_parallel * B_theta / (B * r)
    
    # Toroidal drift (vertical in R-z plane)
    v_drift = (m * v_parallel**2 + mu * B) / (e * B * R0)
    dr_dt = -v_drift * np.sin(theta)
    dtheta_dt += -v_drift * np.cos(theta) / r
    
    return [dr_dt, dtheta_dt, dphi_dt]

# Initial conditions
r0 = 0.5
theta0 = 0.0
phi0 = 0.0
y0 = [r0, theta0, phi0]

t_span = (0, 1e-4) # seconds
t_eval = np.linspace(0, 1e-4, 5000)

sol = solve_ivp(guiding_center_drift, t_span, y0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-10)

R = R0 + sol.y[0] * np.cos(sol.y[1])
Z = sol.y[0] * np.sin(sol.y[1])

plt.figure(figsize=(8, 8))
plt.plot(R, Z, 'b-', linewidth=1.5, label='Guiding Center Orbit')
plt.plot(R0, 0, 'r+', markersize=12, label='Magnetic Axis')
circle = plt.Circle((R0, 0), 1.0, color='gray', fill=False, linestyle='--', label='Wall')
plt.gca().add_patch(circle)
plt.title('Proton Guiding Center Orbit in a Tokamak')
plt.xlabel('Major Radius R (m)')
plt.ylabel('Vertical Z (m)')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.savefig('tokamak_orbit.png')
